# Sprint 5 — The Conductor (Layer A + Adapters + Orchestration)

Demo of the ImmunoSense Conductor: the hub-and-spoke orchestrator that turns
5 independent agents into 1 patient-day evaluation.

## What Sprint 5 built

- **`immunosense/events/`** — Layer A immutable event log (NDJSON), the 6h
  bucket grid, the `PatientBucket` carrier, and cadence-relative freshness.
- **`immunosense/adapters/`** — thin per-agent adapters (translate + isolate
  errors), reusing the existing `AgentOutput` from `agents.base`.
- **`immunosense/conductor/`** — the orchestration flow, per-agent quality
  scoring, and the 4-level confidence aggregation.

## Challenges implemented

| Challenge | What lands here |
|---|---|
| 1 — Temporal granularities | Layer A events + 6h UTC buckets |
| 7 — Missing data | per-agent quality x freshness, 4 confidence levels |
| 9 — Real-time vs batch | sync per-bucket eval + flare-button override |

## What is still a stub (Sprint 6)

`fusion/` (Bayesian probability, corroboration patterns, severity composite)
and `decision/` (when to call TFM / alert) are wired but return `None`/empty.
The report's inference fields stay empty until Sprint 6 fills them in.

In [ ]:
# Section 1: Imports
from datetime import datetime, timezone, timedelta

from immunosense.events import (
    EventLog, BucketBuilder, PatientBucket, AgentData, EventType,
)
from immunosense.adapters import AdapterRegistry
from immunosense.conductor import Conductor

# Real agents (Sprints 1-4)
from immunosense.agents.biomarker.agent import BiomarkerAgent
from immunosense.agents.symptoms_mood.agent import SymptomsMoodAgent
from immunosense.agents.symptoms_mood.types import DailySymptomMoodSummary

print("Sprint 5 Conductor stack imported.")

In [ ]:
# Section 2: Wire the registry with real agents
bio = BiomarkerAgent()
sym = SymptomsMoodAgent()

registry = AdapterRegistry.from_agents([bio, sym])
print("Registered adapters:", registry.agent_ids)
print(f"  (In production all 5 agents register; this demo uses 2.)")

In [ ]:
# Section 3: Set up Layer A event log + Conductor
import tempfile, pathlib
EVENT_ROOT = pathlib.Path(tempfile.mkdtemp(prefix="immunosense_events_"))
log = EventLog(EVENT_ROOT)
conductor = Conductor(registry=registry, event_log=log)
print("Layer A event log at:", EVENT_ROOT)
print("Conductor ready.")

In [ ]:
# Section 4: Build a PatientBucket (Option B - caller provides domain objects)
ts = datetime(2026, 5, 27, 14, 30, tzinfo=timezone.utc)   # 14:30 UTC -> T2
bucket = BucketBuilder.bucket_for("patient001", ts)
print("Bucket:", bucket.bucket_id, "window", bucket.start.time(), "->", bucket.end.time(), "UTC")

pb = PatientBucket(bucket=bucket)

# Symptoms agent: a real DailySymptomMoodSummary
summary = DailySymptomMoodSummary(
    date="2026-05-27", patient_id="patient001", disease="SLE",
    fatigue=7.0, joint_pain=6.0, brain_fog_severity=5.0, gi_distress=2.0,
    skin_severity=4.0, sleep_severity=6.0, energy_severity=6.0, wellness_severity=6.0,
    phq8_score=10.0, gad7_score=8.0,
)
pb.add(AgentData("agent5_symptoms_mood", summary, produced_at=ts))

# Biomarker agent: a weekly reading from 3 days ago (still fresh for weekly cadence)
three_days_ago = ts - timedelta(days=3)
pb.add(AgentData(
    "agent1_biomarker",
    {"demographics": {"age": 45, "sex": 2, "bmi": 27.0},
     "reading": {"CRP": 8.0, "ESR": 30.0}},
    produced_at=three_days_ago,
))

print("PatientBucket assembled. Reporting agents:", pb.reporting_agents)

In [ ]:
# Section 5: Evaluate the bucket
report = conductor.evaluate_bucket(pb)

print("="*64)
print("CONDUCTOR REPORT")
print("="*64)
print(f"Patient:           {report.patient_id}")
print(f"Bucket:            {report.bucket_id}")
print(f"Trace ID:          {report.trace_id}")
print(f"Reporting agents:  {report.reporting_agents}")
print(f"Confidence level:  {report.confidence_level.value.upper()}")
print(f"Overall quality:   {report.overall_quality:.3f}")
print()
print("Per-agent quality (confidence x freshness):")
for aid, q in sorted(report.agent_quality.items()):
    flag = "" if q.reported else "  (absent)"
    print(f"  {aid:26s} reported={q.reported} ok={q.ok} "
          f"raw={q.raw_confidence:.2f} fresh={q.freshness:.2f} -> quality={q.quality:.2f}{flag}")
print()
print("Sprint 6 stubs (currently no-ops):")
print(f"  flare_probability:  {report.flare_probability}")
print(f"  matched_patterns:   {report.matched_patterns}")
print(f"  severity_composite: {report.severity_composite}")
print(f"  decision reasons:   {report.decision.reasons}")
if report.errors:
    print("\nErrors:", report.errors)

In [ ]:
# Section 6: Inspect Layer A — the immutable audit trail
events = log.read_bucket("patient001", bucket.bucket_id)
print(f"Layer A recorded {len(events)} events for this bucket:\n")
for e in events:
    agent = f" [{e.agent_id}]" if e.agent_id else ""
    print(f"  {e.event_type.value:14s}{agent:28s} quality={e.quality:.2f}  trace={e.trace_id}")

# Confirm all events share one trace id (the whole evaluation is one thread)
trace_ids = {e.trace_id for e in events}
print(f"\nAll {len(events)} events share a single trace id: {len(trace_ids) == 1}")

In [ ]:
# Section 7: Freshness is relative to each agent's cadence (Challenge 7)
from immunosense.events import freshness_weight

ref = bucket.end
for cadence in ["1hr", "6hr", "daily", "weekly"]:
    print(f"  cadence={cadence:7s}: ", end="")
    for age_days in [0, 1, 3, 7]:
        produced = ref - timedelta(days=age_days)
        w = freshness_weight(cadence, produced, ref)
        print(f"{age_days}d->{w:.2f}  ", end="")
    print()
print()
print("Read this row-wise: a 3-day-old reading is ~0.66 fresh for a WEEKLY")
print("biomarker agent but ~0.00 for an HOURLY wearable agent. Same age,")
print("different meaning - exactly what Challenge 7 requires.")

In [ ]:
# Section 8: Flare-button override (Challenge 9 - critical event bypasses schedule)
report2 = conductor.on_flare_button(pb, severity=0.95)

flare_events = [e for e in log.read_bucket("patient001", bucket.bucket_id)
                if e.event_type == EventType.FLARE_BUTTON]
print(f"Flare button pressed (severity=0.95).")
print(f"  FLARE_BUTTON events logged: {len(flare_events)}")
print(f"  bucket.flare_button now:    {pb.flare_button}")
print(f"  Re-evaluation confidence:   {report2.confidence_level.value}")
print()
print("The flare button bypasses the 6h schedule: it logs an immutable")
print("FLARE_BUTTON event, then forces an immediate re-evaluation.")

In [ ]:
# Section 9: Error isolation - one bad agent does not crash the bucket
bad_pb = PatientBucket(bucket=bucket)
bad_pb.add(AgentData("agent1_biomarker", {"malformed": "input"}, produced_at=ts))  # bad
bad_pb.add(AgentData("agent5_symptoms_mood", summary, produced_at=ts))             # good

bad_report = conductor.evaluate_bucket(bad_pb)
print("One agent fed malformed input, the other is fine:\n")
for aid, q in sorted(bad_report.agent_quality.items()):
    if q.reported:
        print(f"  {aid:26s} ok={q.ok}  quality={q.quality:.2f}")
print(f"\nErrors captured (not raised): {len(bad_report.errors)}")
for err in bad_report.errors:
    print(f"  - {err[:80]}")
print("\nThe good agent still ran. The failure became a zero-quality")
print("contribution + an AGENT_ERROR event in Layer A. No crash.")

## Sprint 5 complete

The Conductor orchestration skeleton is live:

- **Layer A** records every agent output, error, flare button, and bucket
  evaluation as immutable NDJSON events sharing one trace id.
- **Adapters** translate a `PatientBucket` into each agent's input shape and
  isolate failures so one bad agent never crashes a bucket.
- **Quality scoring** adjusts each agent's self-reported confidence by a
  freshness weight keyed to that agent's cadence (Challenge 7).
- **Confidence aggregation** collapses the per-agent scores into one of four
  levels; INSUFFICIENT is the safety floor.
- **Flare-button override** bypasses the 6h schedule for critical events
  (Challenge 9).

### Next: Sprint 6

The `fusion/` and `decision/` stubs get filled in:
- `statistical_fusion` — Bayesian flare probability (Challenge 3 Phase 1)
- `corroboration` — semantic cross-agent patterns (Phase 2)
- `risk_engine` — severity composite for the UI (Phase 4)
- `decision_maker` — when to call the TFM / raise alerts

All of which consume the Layer A events and confidence levels this sprint
established.